# Exercise: produce_03 — streaming producer & anomalies

**Goal:** continuously stream IoT data, then inject anomalies that a downstream consumer can detect.

**How to use this notebook**
1. Open `exercise_consume_01.ipynb` and run the *Live streaming* cell first
2. Then run the streaming producer below — events should appear live in the consumer
3. Stop the producer with the **■** button when you're done

In [ ]:
from confluent_kafka import Producer
from datetime import datetime
import json, time, random

producer = Producer({'bootstrap.servers': 'redpanda:29092', 'client.id': 'streaming-producer'})

houses  = ['haus_a', 'haus_b', 'haus_c', 'haus_d']
sensors = {
    'strom':  {'unit': 'kWh',   'min': 0.5, 'max': 15.0},
    'wasser': {'unit': 'Liter', 'min': 0.1, 'max': 50.0},
}
print('Ready.')

## Step 1 — continuous stream (1 event every 0.5–2s)

**Task:** complete the event dict (`sensor`, `haus`, `wert`, `einheit`, `timestamp`).

In [ ]:
count = 0
print(f'{"Time":>10} | {"#":>4} | {"Topic":>6} | {"Key":>7} | {"Value":>8}')
print('-' * 50)

try:
    while True:
        house = random.choice(houses)
        topic = random.choice(list(sensors.keys()))
        cfg   = sensors[topic]
        value = round(random.uniform(cfg['min'], cfg['max']), 2)

        event = json.dumps({
            # TODO: fill in the event fields:
            #   'sensor', 'haus', 'wert', 'einheit', 'timestamp'
        })

        producer.produce(topic, key=house.encode(), value=event.encode())
        producer.flush()

        count += 1
        ts = datetime.now().strftime('%H:%M:%S')
        print(f'{ts:>10} | {count:>4} | {topic:>6} | {house:>7} | {value:>8.2f}')
        time.sleep(random.uniform(0.5, 2.0))

except KeyboardInterrupt:
    print(f'\nStopped. Total events sent: {count}')

## Task A — burst mode

Send **50 events as fast as possible** (no `sleep`). What happens in the consumer — does it lag?

In [ ]:
# TODO: send 50 events as fast as possible, then producer.flush() once at the end


## Task B — anomaly simulation

Real sensor streams contain **outliers** — e.g. a defective sensor reporting a huge value.

**Task:** send 30 events where ~20% are anomalies (value ≥ 3× the normal max). Add an `'anomaly': True` flag in the JSON.

**After running:** open Redpanda Console → find a message with `"anomaly": true`.
Then continue with `exercise_consume_03.ipynb` to detect them automatically.

In [ ]:
# TODO: send 30 events; ~20% should have wert > 3*max with 'anomaly': True
# Print '<-- ANOMALY' next to anomalous rows.
